# The Bayon face, one face at a time

An homage to home. Head-on, the cube tessellation shows a smiling face of the Bayon, Angkor's temple of
faces. Each of its 25 faces is painted separately, from its own prompt. Tilted toward any axis, the face
breaks apart into that axis's paintings.

**How it works**

1. **The study** sets the composition: a flat geometric painting of the face, made of polygons
   (`cubism/bayon-study.svg`). You place the model on it.
2. **Each face takes its piece of the study**, unsheared into an upright square: the polygons' edges and
   each polygon's target shade (how light or dark it should be).
3. **Painting:** Stable Diffusion XL paints a face from its prompt, starting from the shades and following
   the edges with a canny ControlNet. Every painting is saved as a numbered **candidate**.
4. **Finishing:** each polygon's lightness is moved most of the way to its target, keeping the painting's
   colors and texture, and the edges are drawn on top in thick black. So a chunk can be saffron or jade,
   but it's as light or dark as the study says, and the face stays legible.
5. **Keeping:** when you like a candidate, keep it. Kept faces are never repainted, so you can try new
   prompts on the rest as often as you like.

Candidates are saved raw, before finishing, so you can change the finishing settings at any time without
repainting anything.

Everything lives in one folder on your Google Drive, so it survives Colab disconnects:

```
MyDrive/folded-illusions/bayon-study/
  candidates/face_07/003.png .json   every painting of face 7 (raw), with the prompt, seed and settings
  kept/face_07.png .json             the one you kept
  finished/face_07.png               face 7 finished, at full size
  board.gif
```

The loop: edit `PROMPTS`, paint a few faces, look, keep the good ones, repeat. The board at the end shows
the whole model with what you've kept so far.

**Like cubism.ipynb, this is virtual:** ideal square faces, no visibility map from real paper yet.

## Setup

Use a **GPU runtime** for painting (Runtime → Change runtime type → T4). Placing the photo and the board
run on CPU too. The first cell asks to connect Google Drive.

In [ ]:
import sys, os, subprocess
from pathlib import Path

BRANCH = "main"                  # the repo branch with bayon.py and the study
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("folded-illusions").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/rath-hy/folded-illusions.git"], check=True)
    sys.path.insert(0, "folded-illusions/cubism")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/folded-illusions/bayon-study")
    STUDY_FILE = Path("folded-illusions/cubism/bayon-study.svg")
else:
    sys.path.insert(0, ".")
    WORK = Path("bayon-work/study")   # next to the notebook, when run locally
    STUDY_FILE = Path("bayon-study.svg")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import bayon as b

store = b.Store(WORK)
print("working folder:", WORK)

In [ ]:
%%capture
# Painting needs these; skip on CPU-only runs.
!pip install -q -U diffusers transformers accelerate

## The study

The SVG is cropped first (`CROP`: fractions of its width or height cut from each side) to trim the empty
border. Its filled polygons become chunks and its lines become extra edges; hatching and the canvas
texture are ignored. Left: the study as drawn. Right: what the faces will hold, each chunk's target
shade in gray, with the edges as thick as they'll be.

To change the composition, edit the SVG (any vector editor) and rerun from here. Faces you kept keep
their paintings, but check them on the board: their edges and shades come from the new study.

In [ ]:
CROP = dict(left=1/8, right=1/8, top=0, bottom=1/8)
study = b.load_study(STUDY_FILE, CROP)
print(f"{len(study.chunks)} chunks, {len(study.lines)} extra lines, {study.width:.0f} x {study.height:.0f} units")

fig, axes = plt.subplots(1, 2, figsize=(12, 9))
axes[0].imshow(b.draw_study(study, 1.5, original=True)); axes[0].set_title("the study, cropped")
axes[1].imshow(b.draw_study(study, 1.5)); axes[1].set_title("target shades and edges")
for ax in axes:
    ax.axis("off")
plt.show()

## Finishing

How finished faces look. Change these any time: nothing is repainted, the board just updates.

- `shade`: how far each chunk's lightness moves toward the study's, from 0 (not at all) to 1 (all the
  way). At 0.8 a chunk ends up close to its target but keeps a little of the painting's own light.
- `edge_width`: the edges' thickness in pixels on a 1024-pixel face.
- `edge_color`: one color for every edge.
- `shades`: the darkest and lightest targets, as CIELAB lightness (0 black, 100 white). The study's
  near-black chunks map to the first, so they're dark but still show color, and black edges stand out
  on them.

In [ ]:
FINISH = dict(shade=0.8, edge_width=24, edge_color=(0, 0, 0), shades=(15, 92))

## Place the model

`FIT` puts the model on the study, head-on: `x` and `y` are its center as fractions of the study's width
and height, and `width` is how much of the study's width it spans. Left: the faces on the study. Right:
the model head-on with every face showing its piece of the study, as unpainted faces will look on the
board.

Faces are numbered 1 to 25 in reading order, top to bottom and left to right. Any part of a face that
falls outside the study is black, so keep the outlines inside it.

In [ ]:
FIT = dict(x=0.5, y=0.5, width=1.0)

tex = [b.to_texture(n, b.study_face(study, n, FIT, 512, FINISH), 256) for n in b._in_faces_order()]
fig, axes = plt.subplots(1, 2, figsize=(14, 9))
axes[0].imshow(b.outlined_photo(b.draw_study(study, 1.5, FINISH, FIT), FIT)); axes[0].set_title("the faces on the study")
axes[1].imshow(b.numbered(b.render(tex, size=768), 768)); axes[1].set_title("the model head-on")
for ax in axes:
    ax.axis("off")
plt.show()

**One face up close.** Each face is painted as an upright square: as you'd see it with the model tilted
straight toward that face's axis, turned so its top is as close to up as it can be. Head-on it's
sheared, which is why its piece looks slanted on the study. Left to right: the study's own colors, the
edges the painter follows, and the shades it starts from. Change `LOOK` to see another face.

In [ ]:
LOOK = 17

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(b.face_photo(b.draw_study(study, 2.0, FINISH, FIT, original=True), LOOK, FIT, 512))
axes[0].set_title(f"face {LOOK}: the study")
axes[1].imshow(b.face_lines(study, LOOK, FIT, 512, 4), cmap="gray"); axes[1].set_title("edges for the painter")
axes[2].imshow(b.face_shades(study, LOOK, FIT, 512, FINISH["shades"])); axes[2].set_title("shades it starts from")
for ax in axes:
    ax.axis("off")
plt.show()

## Prompts

One prompt per face. `STYLE` is added to every prompt, which keeps 25 separate paintings looking like
one piece; `NEGATIVE` steers away from things on every face. The starters below are placeholders: make
them yours.

Changing a prompt never touches a face you've kept. Every candidate records the prompt that made it.

In [ ]:
STYLE = ("a synthetic cubist painting, bold flat colors, sharp geometric shapes with hard straight edges, "
         "saffron, lotus pink, jade green, temple gold, deep red and indigo")
NEGATIVE = "blurry, soft edges, gradients, photo, realistic, muddy colors, text, letters, watermark, signature"

PROMPTS = {
    1: "the five lotus towers of Angkor Wat at sunrise",
    2: "an apsara dancer carved in sandstone",
    3: "a pink lotus in bloom",
    4: "a seven-headed naga serpent",
    5: "sugar palm trees against the sky",
    6: "the Mekong river at dusk with long fishing boats",
    7: "a monk in saffron robes",
    8: "a red and white checked krama scarf",
    9: "a golden seated Buddha",
    10: "rice paddies with a water buffalo",
    11: "a Khmer temple roof with curling finials",
    12: "houses on stilts in a floating village",
    13: "a tuk-tuk on a red dirt road",
    14: "Garuda with spread wings",
    15: "fishermen casting nets on the Tonle Sap lake",
    16: "jasmine flower offerings",
    17: "the churning of the ocean of milk, a temple bas-relief",
    18: "a Khmer silk weaving loom",
    19: "green pepper vines in Kampot",
    20: "the water festival boat races",
    21: "a lotus pond with lily pads",
    22: "a hamsa, the sacred goose",
    23: "banyan tree roots over temple stones",
    24: "candles and incense at a shrine",
    25: "a stone lion guarding a temple stair",
}
assert sorted(PROMPTS) == b.NUMBERS, "one prompt per face, 1 to 25"

labels = {n: ("KEPT" if n in store.kept() else "") for n in b.NUMBERS}
plt.figure(figsize=(8, 8))
plt.imshow(b.numbered(b.render(b.board(store, study, FIT, settings=FINISH), size=768, light=b.LIT), 768, labels=labels))
plt.axis("off"); plt.title("faces so far"); plt.show()
for n in b.NUMBERS:
    print(f"{n:>2} {'kept' if n in store.kept() else '    '}  {PROMPTS[n]}")

## Paint

`PAINT` is which faces to paint this run: a list of face numbers, `"unkept"` for every face you haven't
kept yet, or `"all"`. A face you list by number is painted even if it's kept, which only adds
candidates; your kept choice stays until you change it.

Each face gets `PER_FACE` new candidates with fresh random seeds (or set `SEEDS` to repeat exact ones).
On a T4, each painting takes about a minute, so a full `"unkept"` round of 25 faces at 2 each is about
50 minutes. Start with a few.

- `strength`: how far the painter may stray from the shades it starts from. Lower keeps the study's
  light and dark (and its grayness); higher gives the prompt more room. Finishing fixes the shades
  either way.
- `control_scale`: how strictly the painting follows the edges.
- `control_width`: how thick the painter sees the edges (on a 1024-pixel face). Thin, like the canny
  edges this ControlNet was trained on; the thick edges are drawn later, when finishing.
- `size`: 1024 is SDXL's native size. 768 is about twice as fast, for quick looks.

Each face shows its raw candidates on top and the same candidates finished below.

In [ ]:
PAINT = [13, 17, 18]
PER_FACE = 2
SEEDS = None                      # or a list, e.g. [0, 1], to repeat exact seeds
SETTINGS = dict(strength=0.8, control_scale=0.6, control_width=4, guidance=7.0, steps=30, size=1024)

if "painter" not in globals():
    painter = b.load_painter()

rng = np.random.default_rng()
for n in b.todo(store, PAINT):
    seeds = SEEDS if SEEDS is not None else [int(s) for s in rng.integers(0, 2**31, PER_FACE)]
    made = b.paint(painter, store, study, n, PROMPTS[n], seeds, STYLE, NEGATIVE, FIT, SETTINGS, FINISH["shades"])
    fig, axes = plt.subplots(2, len(made) + 1, figsize=(4 * (len(made) + 1), 8.6), squeeze=False)
    axes[0, 0].imshow(b.face_photo(b.draw_study(study, 2.0, FINISH, FIT, original=True), n, FIT, 512))
    axes[0, 0].set_title(f"face {n}: the study")
    axes[1, 0].imshow(b.study_face(study, n, FIT, 512, FINISH)); axes[1, 0].set_title("its target")
    for k, c in enumerate(made):
        pic, info = store.candidate(n, c)
        axes[0, k + 1].imshow(pic); axes[0, k + 1].set_title(f"candidate {c}, raw")
        axes[1, k + 1].imshow(b.finish(pic, study, n, FIT, FINISH)); axes[1, k + 1].set_title(f"candidate {c}, finished")
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(PROMPTS[n], y=1.02)
    plt.show()

## Keep

`KEEP` maps a face to the candidate you want: `{13: 2}` keeps face 13's candidate 2. Keeping again
replaces your choice. `UNKEEP` sends faces back to unpainted (their candidates stay).

To look back at every candidate of a face, set `BROWSE`. The kept one is outlined in red. `FINISHED`
shows them as they'd end up; set it to `False` to see them raw.

In [ ]:
KEEP = {}                         # e.g. {13: 2, 17: 1}
UNKEEP = []

for n, c in KEEP.items():
    info = store.keep(n, c)
    print(f"kept face {n}: candidate {c}, seed {info['seed']}, \"{info['prompt']}\"")
for n in UNKEEP:
    store.unkeep(n)
    print(f"face {n} is unpainted again")
print(f"{len(store.kept())} of 25 faces kept")

In [ ]:
BROWSE = 13
FINISHED = True

sheet = b.contact_sheet(store, BROWSE, finished=(lambda pic: b.finish(pic, study, BROWSE, FIT, FINISH)) if FINISHED else None)
if sheet is None:
    print(f"face {BROWSE} has no candidates yet")
else:
    plt.figure(figsize=(14, 14 * sheet.height / sheet.width)); plt.imshow(sheet); plt.axis("off")
    plt.title(f"face {BROWSE}: every candidate"); plt.show()

## The board

The whole model with what you've kept, finished. Faces not yet kept show the study in gray. Head-on
(lit as paper would be) is the artwork; each tilt is a jumble of one axis's paintings. Saves the board
as `board.gif` and every kept face, finished at full size, to `finished/`.

In [ ]:
tex = b.board(store, study, FIT, settings=FINISH)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(b.render(tex, size=512, light=b.LIT)); axes[0].set_title("head-on")
for k in range(3):
    axes[k + 1].imshow(b.render(tex, t=1.0, axis=k, size=384)); axes[k + 1].set_title(f"tilted to {b.AXES[k]}")
for ax in axes:
    ax.axis("off")
plt.show()

plt.figure(figsize=(9, 9)); plt.imshow(b.render(tex, size=900, light=b.LIT)); plt.axis("off"); plt.show()

from IPython.display import Image as ShowImage
b.save_gif(tex, WORK / "board.gif")
print("finished faces saved:", b.save_finished(store, study, WORK / "finished", FIT, FINISH))
ShowImage(filename=str(WORK / "board.gif"))

## What this leaves out

- **The real visibility map.** Like the guitar, this paints ideal cube faces. Printing needs the map from
  the folded sheet to each face, and the model's rim has flat triangles we haven't painted.
- **Lighting.** Real light makes the three axes' faces different brightnesses (`LIT` imitates it), so a
  face's colors shift by axis when folded. We could pre-brighten the darker axes before printing.
- **Seams.** Neighboring faces are painted separately, so their colors don't continue across borders,
  only the study's edges and shades do. That suits cubism, but a shared `STYLE` palette matters.
- **Color at the extremes.** A very light or very dark chunk can't hold a strong color, so finishing
  mutes it there. Narrowing `shades` keeps more color at the cost of contrast.